# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shenone77/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

A binary outcome supplies the score for a ranked editorial review queue.

## Setup

Open this notebook from the repository or its Colab badge. The analysis uses FlyRank's bundled anonymized sample.

In [1]:
from pathlib import Path
import subprocess
import sys

if "google.colab" in sys.modules:
    root = Path("/content/flyrank-ml-internship")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Shenone77/flyrank-ml-internship.git", str(root)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2", "scikit-learn>=1.4"], check=True)
else:
    start = Path.cwd().resolve()
    root = next(
        (path for path in (start, *start.parents) if (path / "work/notebooks/refresh_triage.py").is_file()),
        None,
    )
    if root is None:
        raise FileNotFoundError("Open the notebook inside a clone of the assignment repository.")

sys.path.insert(0, str(root / "work/notebooks"))
from refresh_triage import load_pages

pages = load_pages(root)
print(f"Loaded {len(pages):,} eligible pages from the FlyRank starter sample.")

Loaded 13,412 eligible pages from the FlyRank starter sample.


## 1. My lane as an ML task

The operational task is ranking. I will fit a binary classifier and sort pages by its estimated likelihood of material decline. The editor uses the first 50 to investigate visibility loss. The initial comparator is a transparent exposure rule with a small boost for low prior CTR.

## 2. Target or proxy

Let I be impressions in the 30 days before the cutoff and J impressions in the following 30 days. Keep I ≥ 300. Set the label to 1 when I − J ≥ 150 and (I − J) / I ≥ 30%, and to 0 otherwise. This label uses measured impressions and reflects a review threshold. A decline can have several causes, including indexing, seasonality, or changed demand.

**Feature window.** Prior impressions and prior clicks end at the cutoff. CTR uses clicks divided by impressions from that same window. Age at the cutoff is the snapshot's days-since-creation minus 30. The outcome window enters only through the label. The 90-day rolling totals, snapshot freshness, trend fields, and product flags stay outside the predictors. The prior count is also the label's reference level, so exposure can affect measured risk mechanically.

## 3. Success metric

Use **Precision@50** on the same held-out clients for both methods. Each true positive in the top 50 is one editor slot spent on a page that later crossed the material-loss thresholds. Compare with the held-out base rate to show the value of selecting 50 eligible pages evenly at random. Set the primary budget before fitting and report Precision@20 as a smaller-budget check. An improvement is useful only if it survives a later snapshot and the actual editor's workflow.

## 4. The unit of analysis as a dataframe

Here are six genuine page rows from the loaded sample, with identifying fields omitted and the displayed features binned or rounded. The label is computed per row from the observed follow-up window.

In [2]:
import pandas as pd
from refresh_triage import FEATURES, SEED

example = pages.sample(n=6, random_state=SEED)
preview = pd.DataFrame({
    "Prior impressions (band)": pd.cut(
        example["impressions_prev_30d"], [299, 999, 2999, float("inf")],
        labels=["300-999", "1000-2999", "3000+"],
    ),
    "Prior CTR (%)": example["prior_ctr_pct"].round(1),
    "Decision age (30-day bands)": example["age_at_cutoff_days"].floordiv(30).mul(30),
    "Material drop (0/1)": example["material_drop"].astype(int),
})
display(preview.reset_index(drop=True))
assert pages.loc[:, list(FEATURES)].notna().all().all()
print(f"{len(pages):,} eligible page rows and {pages['material_drop'].sum():,} positive labels.")

,Prior impressions (band),Prior CTR (%),Decision age (30-day bands),Material drop (0/1)
0,1000-2999,0.3,360,0
1,3000+,0.8,390,0
2,3000+,0.2,120,0
3,1000-2999,0.0,240,1
4,300-999,0.0,120,0
5,300-999,0.0,60,1


13,412 eligible page rows and 6,643 positive labels.


## 5. When the model earns its place

The rule values exposure directly. A model can learn whether exposure, CTR, and page age jointly make a drop more likely within this population, which may give a different ordering. Label quality, a stable evaluation, and the same action budget matter more than a more complex model. If the rule is equally effective in a representative evaluation, its simplicity makes it preferable for the editor.

## Self-check

The unit, target, review action, budget, and split are specified. The public preview excludes identifiers. The delayed label and the pre-cutoff feature list are documented separately.